# 11. 핸드오프와 라우팅

핸드오프는 호출한 쪽으로 결과를 돌려주지 않고 다음 담당자에게 제어권을 넘긴다. 에이전트를 도구로 호출하는 방식과 핸드오프의 실패 조건을 제어권 관점에서 비교한다.

### 공통 설정

`1`과 같은 블록이다. 라우팅과 인수인계 판정에는 하위 모델을 쓴다.

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50)}
USAGE = []

def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 에이전트를 도구로 호출하기와 핸드오프

다른 에이전트에게 일을 맡기는 방법은 둘이다. 그 에이전트를 도구처럼 불러 결과만 받아 오는 방식(agent as tool)과, **제어권을 통째로 넘기는 핸드오프**다.

| | 에이전트를 도구로 | 핸드오프 |
|---|---|---|
| 제어 | 부른 쪽이 계속 쥔다 | 받은 쪽으로 넘어간다 |
| 결과 | 호출한 자리로 돌아온다 | 받은 쪽이 사용자에게 답한다 |
| 컨텍스트 | 그대로 이어진다 | 필요한 정보만 골라 넘긴다 |
| 실패 조건 | 결과가 커져 컨텍스트가 부푼다 | 정보가 부족하면 받은 쪽이 맥락을 잃는다 |

이 노트북은 핸드오프 쪽만 돌린다. 에이전트를 도구로 부르는 쪽은 `10`에서 오케스트레이터가 워커를 부르고 결과를 받던 방식이다.

### 도구와 루프

도구는 세기와 읽기 둘이다.

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")


def grep_docs(pattern):
    return [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]


def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(f"'{title}' 문서가 없다")
    return f.read_text(encoding="utf-8")[:4000]

In [3]:
def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


GREP = spec("grep_docs", "낱말이 든 문서의 제목과 등장 횟수를 돌려준다.",
            {"pattern": {"type": "string", "description": "찾을 낱말"}})
READ = spec("read_doc", "문서 한 편의 앞부분을 돌려준다.",
            {"title": {"type": "string", "description": "문서 제목"}})
IMPL = {"grep_docs": grep_docs, "read_doc": read_doc}

`9`의 코어 블록이다. 전문가마다 지시문과 도구가 다르므로 설정 `cfg` 하나가 곧 전문가 하나다. `history`로 앞 대화를 이어받고, 끝난 대화를 네 번째 값으로 돌려준다.

In [ ]:
CONFIG = {"model": MODEL, "instructions": "도구로 확인한 것만 답한다.", "tools": [GREP, READ], "impl": IMPL,
          "max_turns": 5, "max_tokens": 200000, "max_seconds": 180, "gated": [],
          "gate_error": "error: 승인이 필요하다.", "tool_error": "error: {e}"}


def call_tool(name, args, cfg, approve):
    if name in cfg["gated"] and not approve:
        return cfg["gate_error"]
    try:
        return json.dumps(cfg["impl"][name](**args), ensure_ascii=False)
    except Exception as e:
        return cfg["tool_error"].format(e=e)


def run(task, cfg=CONFIG, history=None, approve=False):
    messages = list(history or []) + [{"role": "user", "content": task}]
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, cfg["max_turns"] + 1):
        r = keep(client.responses.create(model=cfg["model"], instructions=cfg["instructions"],
                                         input=messages, tools=cfg["tools"], store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})
        if not calls:
            return r.output_text, "완료", trace, messages + [{"role": "assistant", "content": r.output_text}]
        if used > cfg["max_tokens"]:
            return None, "토큰 예산", trace, messages
        if time.perf_counter() - t0 > cfg["max_seconds"]:
            return None, "시간 예산", trace, messages
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), cfg, approve)}
                     for c in calls]
    return None, "반복 예산", trace, messages

### 전문가 구성

같은 모델에 서로 다른 지시와 도구를 할당한다. 나머지는 `CONFIG`를 그대로 물려받는다.
</br>실무에서는 각 에이전트들이 모델도 달라지는 경우가 많음
</br>툴이 다양해지면, 도구들을 묶어서 특화된 agent로 관리하는게 적절

In [5]:
AGENTS = {
    "counter": {**CONFIG, "tools": [GREP],
                "instructions": "세는 질문만 맡는다. grep_docs 로 세고 숫자로 답한다."},
    "reader": {**CONFIG, "tools": [READ],
               "instructions": "한 문서의 내용을 묻는 질문만 맡는다. read_doc 으로 읽고 답한다."},
    "refuser": {**CONFIG, "tools": [],
                "instructions": "이 코퍼스는 대한민국 임시정부 문서 50편이다."
                                " 범위 밖 질문에는 답할 수 없다고만 말한다."},
}

라우터는 구조화 출력으로 셋 중 하나를 고른다. `Literal`이 이름을 세 개로 묶는다.

In [7]:
class Route(BaseModel):
    agent: Literal["counter", "reader", "refuser"]
    why: str


GUIDE = ("질문을 셋 중 하나로 보낸다. 문서 수나 빈도를 세는 질문은 counter,"
         " 특정 문서의 내용을 묻는 질문은 reader,"
         " 이 코퍼스(대한민국 임시정부 문서 50편) 밖의 질문은 refuser.")

질문 여섯 개에 기대하는 담당자를 붙여 둔다. (Ground Truths)

In [8]:
QUESTIONS = [("counter", "한국독립당은 몇 편의 문서에 나오는가"),
             ("counter", "의열단이 가장 많이 나오는 문서는 무엇인가"),
             ("reader", "한인애국단 문서는 이 단체를 어떻게 설명하는가"),
             ("reader", "윤봉길 문서에 적힌 의거 날짜는 언제인가"),
             ("refuser", "오늘 서울 날씨는 어떤가"),
             ("refuser", "파이썬으로 정렬하는 법을 알려 달라")]

하나를 먼저 보낸다. 담당자와 이유가 같이 온다.

In [9]:
out = keep(client.responses.parse(model=LOWER, store=False, instructions=GUIDE,
                                  input=QUESTIONS[0][1], text_format=Route)).output_parsed
print(out.agent, "|", out.why)

counter | 질문은 코퍼스(대한민국 임시정부 문서 50편) 안에서 특정 단체인 ‘한국독립당’이 몇 편의 문서에 등장하는지 묻고 있으므로 문서 수를 세는 counter 유형이다.


여섯 개를 모두 보내 기대와 맞춘다.

In [10]:
picked = []
for want, q in QUESTIONS:
    out = keep(client.responses.parse(model=LOWER, store=False, instructions=GUIDE,
                                      input=q, text_format=Route)).output_parsed
    picked.append(out.agent == want)
    print(f"{'ok' if out.agent == want else '  '} {out.agent:8} <- {q[:30]}")

print(f"\n라우팅 {sum(picked)}/{len(picked)}")

ok counter  <- 한국독립당은 몇 편의 문서에 나오는가
ok counter  <- 의열단이 가장 많이 나오는 문서는 무엇인가
ok reader   <- 한인애국단 문서는 이 단체를 어떻게 설명하는가
ok reader   <- 윤봉길 문서에 적힌 의거 날짜는 언제인가
ok refuser  <- 오늘 서울 날씨는 어떤가
ok refuser  <- 파이썬으로 정렬하는 법을 알려 달라

라우팅 6/6


### 핸드오프 뒤의 컨텍스트

두 번째 질문이 첫 번째 결과를 참조한다. 한 에이전트가 대화를 이어받으면 연결된다.

In [11]:
Q1 = "의열단이 나오는 문서를 세어 숫자로 답한다"
Q2 = "방금 센 그 문서들 중 첫 번째 것을 읽고 한 문장으로 요약한다"

a1, _, _, hist = run(Q1)
a2 = run(Q2, history=hist)[0]
print("  1 |", str(a1)[:50])
print("  2 |", str(a2)[:70])

  1 | 10
  2 | 김구는 일제강점기 대한민국 임시정부의 핵심 인물로 활동하며 한인애국단을 조직해 독립운동을 이끈 독립운동가이자 정치인이다.


다른 전문가에게 컨텍스트 없이 넘기면 연결이 끊긴다.

In [12]:
b1, _, _, bhist = run(Q1, AGENTS["counter"])
b2 = run(Q2, AGENTS["reader"])[0]
print("  1 |", str(b1)[:50])
print("  2 |", str(b2)[:70])

  1 | 10
  2 | 어떤 문서들이었는지 이 대화에는 보이지 않아 첫 번째 문서를 특정할 수 없습니다. 첫 번째 문서의 제목을 알려주시면 읽고 한 


### 인수인계 설계

대화 전체를 넘기지 않고 다음 담당자에게 필요한 정보만 구조화한다.

In [13]:
class Handoff(BaseModel):
    이유: str
    사실: list[str]
    다음_할_일: str

인수인계 문서는 하위 모델이 원본에서 뽑는다. 다음 담당자는 그 문서만 앞에 두고 시작한다.

In [14]:
def handoff(source, label):
    brief = keep(client.responses.parse(
        model=LOWER, store=False, text_format=Handoff,
        input=f"앞 담당자의 기록\n{source[:3000]}\n\n다음 담당자가 할 일\n{Q2}\n\n"
              "다음 담당자가 알아야 할 사실만 골라 인수인계한다.")).output_parsed
    note = json.dumps(brief.model_dump(), ensure_ascii=False)
    text = run(Q2, AGENTS["reader"], history=[{"role": "user", "content": "인수인계\n" + note}])[0]
    print(f"{label}")
    print(f"  실은 사실 {brief.사실}")
    print(f"  결과 {str(text)[:70]}")

앞 담당자의 최종 답을 원본으로 쓴다.

In [15]:
handoff(str(a1), "최종 답에서 뽑은 인수인계")

최종 답에서 뽑은 인수인계
  실은 사실 ['현재 제공된 정보에는 ‘문서 1’의 본문이 없다.', '첫 번째 문서를 읽어야 한 문장 요약을 할 수 있다.', '다음 담당자는 첫 번째 문서의 내용 확보 후 요약 작업을 하면 된다.']
  결과 첫 번째 문서의 제목이나 식별자가 없어서 아직 읽을 수 없습니다.  
방금 센 문서들 중 첫 번째 문서의 **제목**을 알려주


앞 담당자의 실행 기록에서 도구 결과만 모아 원본으로 쓴다.

In [16]:
log = "\n".join(m["output"] for m in bhist
                if isinstance(m, dict) and m.get("type") == "function_call_output")
handoff(log, "실행 기록에서 뽑은 인수인계")

실행 기록에서 뽑은 인수인계
  실은 사실 ["다음 담당자가 처리할 문서는 목록의 첫 번째 항목인 '김구'이다.", '해야 할 일은 해당 문서를 읽고 한 문장으로 요약하는 것이다.', '인수인계는 이 작업을 수행하는 데 필요한 최소 정보만 전달하면 된다.']
  결과 김구는 황해도 해주 출신의 독립운동가이자 정치인으로, 대한민국 임시정부의 중심 인물로 활동하며 한인애국단을 조직해 항일 투쟁을


### 실행 기록에서 인수인계 정보 추출

최종 답은 요약 과정에서 다음 담당자에게 필요한 정보를 잃을 수 있다. 실행 기록에는 도구 결과가 남아 있으므로 인수인계에 필요한 사실을 추출할 수 있다.

### 반환 조건

전문가는 범위 밖 질문을 라우터에 반환해야 한다. 처리 여부와 반환 이유를 구조화하면 라우터가 자연어 답변을 다시 해석하지 않아도 된다.

In [18]:
class Reply(BaseModel):
    처리함: bool
    답: str
    되돌리는_이유: str


def ask_expert(name, q):
    a = AGENTS[name]
    text = run(q, {**a, "instructions": a["instructions"] + " 범위 밖이면 처리하지 않았다고 표시한다."})[0]
    return keep(client.responses.parse(
        model=LOWER, store=False, text_format=Reply,
        input=f"담당 {name}\n질문 {q}\n담당자의 답\n{text}\n\n"
              "담당자가 질문을 실제로 처리했는지 판정한다.")).output_parsed

맞는 질문 하나와 범위 밖 질문 둘을 전문가에게 직접 보낸다.

In [20]:
for name, q in [("counter", "한국독립당은 몇 편에 나오는가"),
                ("counter", "윤봉길 문서의 첫 문장은 무엇인가"),
                ("reader", "오늘 서울 날씨는 어떤가")]:
    out = ask_expert(name, q)
    print(f"{name:8} {'처리' if out.처리함 else '반환'}  {(out.답 if out.처리함 else out.되돌리는_이유)}")

counter  반환  질문의 의도는 한국독립당이 등장하는 편 수를 확인하는 것이지만, 담당자의 답변은 근거 없이 숫자만 제시해 질문을 실제로 해결했는지 판단할 수 없습니다.
counter  반환  질문은 '윤봉길 문서의 첫 문장'을 묻고 있지만, 담당자의 답변은 그 내용을 제공하지 않고 처리 거부/미처리 상태만 알립니다.
reader   반환  질문은 오늘 서울 날씨를 묻는 일반 정보 질문으로, 문서 내용에 관한 질문이 아닙니다. 담당자의 답변도 문서 질문만 가능하다고 하여 실제로 처리하지 않았습니다.


### 하나로 잇기

라우팅, 전문가 실행, 반환 판정을 함수 하나로 잇는다. 전문가가 처리하지 않았다고 판정되면 되돌린 이유를 붙여 라우터에 다시 보낸다.
같은 질문이 끝없이 오가지 않도록 넘기는 횟수에 상한을 둔다. refuser 의 거절은 되돌리지 않고 답으로 본다.

In [ ]:
def serve(q, hops=2):
    note = ""
    for _ in range(hops):
        # 라우팅
        route = keep(client.responses.parse(model=LOWER, store=False, instructions=GUIDE,
                                            input=q + note, text_format=Route)).output_parsed
        # 실행과 반환 판정
        reply = ask_expert(route.agent, q)
        if reply.처리함 or route.agent == "refuser":
            return route.agent, reply
        note = f"\n\n앞 담당 {route.agent} 가 처리하지 않고 되돌렸다. 이유: {reply.되돌리는_이유}"  # 다음 라우터에게 전달함
    return route.agent, reply

In [22]:
for q in ["한국독립당은 몇 편에 나오는가", "윤봉길 문서에 적힌 의거 날짜는 언제인가", "오늘 서울 날씨는 어떤가"]:
    agent, reply = serve(q)
    print(f"{agent:8} {'처리' if reply.처리함 else '반환'}  {(reply.답 if reply.처리함 else reply.되돌리는_이유)[:52]}")

counter  처리  16편
reader   처리  문서에 따르면 의거 날짜는 **1932년 4월 29일**입니다.
refuser  반환  질문은 오늘 서울 날씨를 묻는 일반 시사/실시간 정보로, 코퍼스 범위 내 문서 질문이 아니므로


### 선택 기준

| 상황 | 방식 |
|---|---|
| 결과를 받아 같은 주체가 계속 판단한다 | 에이전트를 도구로 호출 |
| 대화의 주체가 바뀐다 | 핸드오프 |
| 독립적인 하위 과제의 결과를 합친다 | 오케스트레이터 |

핸드오프에는 **전달할 정보와 형식을 정한 인수인계 계약**이 필요하다. 대화 전체를 전달해야 한다면 단일 에이전트를 유지하는 편이 비용이 적다.

### 비용 확인

In [23]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")

호출 44회
입력 16,064 토큰   출력 2,315 토큰
누적 $0.0507
